#### ADB merge
ADB PO 2025 pda + ADB AO 2020 + ADB AS 2026 + ADB AM 2026 + ISPRA 2020

In [1]:
import logging
logging.basicConfig(level=logging.INFO)
import geopandas as gpd
import networkx as nx
import pandas as pd

In [2]:
adbpo_file = r"/home/admin_climatecharted_com/data/ADB/AA_pda2025/AA_pda2025_H_M_L/AA_pda2025_H_M_L.shp"
adbpo = gpd.read_file(adbpo_file)
print(adbpo.crs)
print(adbpo.columns)

adbao_file = r"/home/admin_climatecharted_com/data/ADB/adb_ao/ADB_AO_Tiranti/ADB_AO_Tiranti.shp"
adbao = gpd.read_file(adbao_file)
print(adbao.crs)
print(adbao.columns)

adbas_file = r"/home/admin_climatecharted_com/data/ADB/adb_as_2026/ADB-AS_2026_merge_RP_overlay/ADB-AS_2026_merge_RP_overlay.shp"
adbas = gpd.read_file(adbas_file)
print(adbas.crs)
print(adbas.columns)

adbam_file = r"/home/admin_climatecharted_com/data/ADB/adb_am_2026/ADB-AM_2026_merge_RP_overlay/ADB-AM_2026_merge_RP_overlay.shp"
adbam = gpd.read_file(adbam_file)
print(adbam.crs)
print(adbam.columns)

/home/admin_climatecharted_com/miniforge3/envs/ccpy4/lib/python3.14/site-packages/pyogrio/raw.py:200: RuntimeWarning: /home/admin_climatecharted_com/data/ADB/AA_pda2025/AA_pda2025_H_M_L/AA_pda2025_H_M_L.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


EPSG:3035
Index(['fid', 'competentA', 'nomeElIdr', 'sourceOfFl', 'characteri',
       'mechanismO', 'returnPeri', 'determinat', 'legenda', 'RP', 'layer',
       'geometry'],
      dtype='str')


/home/admin_climatecharted_com/miniforge3/envs/ccpy4/lib/python3.14/site-packages/pyogrio/raw.py:200: RuntimeWarning: /home/admin_climatecharted_com/data/ADB/adb_ao/ADB_AO_Tiranti/ADB_AO_Tiranti.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


EPSG:3035
Index(['h', 'RP', 'geometry'], dtype='str')
EPSG:3003
Index(['RP', 'geometry'], dtype='str')
EPSG:32633
Index(['RP', 'geometry'], dtype='str')


In [3]:
target_crs = "EPSG:3035"

adbao = adbao.to_crs(target_crs)
adbas = adbas.to_crs(target_crs)
adbam = adbam.to_crs(target_crs)
# adbpo already in 3035

In [ ]:
adbpo["geometry"] = adbpo.buffer(0)
adbao["geometry"] = adbao.buffer(0)
adbas["geometry"] = adbas.buffer(0)
adbam["geometry"] = adbam.buffer(0)

In [4]:
adbpo["adb"] = "PO"


In [5]:
adbao["adb"] = "AO"


In [6]:
adbas["adb"] = "AS"


In [7]:
adbam["adb"] = "AM"

In [8]:
adbpo = adbpo[['RP', 'adb', 'geometry']]
adbao = adbao[['RP', 'adb', 'geometry']]
adbas = adbas[['RP', 'adb', 'geometry']]
adbam = adbam[['RP', 'adb', 'geometry']]

In [9]:
final_gdf = gpd.GeoDataFrame(
    pd.concat([adbpo, adbao, adbas, adbam], ignore_index=True),
    crs=target_crs
)

In [10]:
output_file = "/home/admin_climatecharted_com/data/ADB/ADB_merged"
final_gdf.to_file(output_file)

INFO:pyogrio._io:Created 837,095 records


#### Clip ISPRA to delim adb ac, adb si, adb sa

In [11]:
delim_adb_sa_file = r"/home/admin_climatecharted_com/data/ADB/delimitazione_distretto_ADB_Sardegna/delimitazione_distretto_ADB_Sardegna.shp"
delim_adb_si_file = r"/home/admin_climatecharted_com/data/ADB/delimitazione_distretto_ADB_Sicilia/delimitazione_distretto_ADB_Sicilia.shp"
delim_adb_ac_file = r"/home/admin_climatecharted_com/data/ADB/delimitazione_distretto_ADB_App_Centrale/delimitazione_distretto_ADB_App_Centrale.shp"


In [ ]:
delim_adb_ac = gpd.read_file(delim_adb_ac_file)
print(delim_adb_ac.crs)
print(delim_adb_ac.columns)
print(delim_adb_ac)

In [12]:
layers = {
    "H": r"/home/admin_climatecharted_com/data/ISPRA/HPH_Mosaicatura_ISPRA_2020_premerge/HPH_Mosaicatura_ISPRA_2020_premerge.shp",
    "M": r"/home/admin_climatecharted_com/data/ISPRA/MPH_Mosaicatura_ISPRA_2020_premerge/MPH_Mosaicatura_ISPRA_2020_premerge.shp",
    "L": r"/home/admin_climatecharted_com/data/ISPRA/LPH_Mosaicatura_ISPRA_2020_premerge/LPH_Mosaicatura_ISPRA_2020_premerge.shp",
}

In [ ]:
gdfs = {k: gpd.read_file(v) for k, v in layers.items()}
gdfs = {k: gdf.assign(geometry=gdf.geometry.buffer(0)) for k, gdf in gdfs.items()}
logging.info("ISPRA - loaded gdfs")

In [ ]:
import geopandas as gpd
import pandas as pd
import logging

def process_adb(delim_gdf, gdfs, adb_name):
    logging.info(f"Processing ADB: {adb_name}")

    # -------------------------
    # 1. CLIP + STANDARDIZE
    # -------------------------
    H = gpd.clip(gdfs["H"], delim_gdf)
    M = gpd.clip(gdfs["M"], delim_gdf)
    L = gpd.clip(gdfs["L"], delim_gdf)

    # Assign attributes
    H["RP"] = 20
    M["RP"] = 100
    L["RP"] = 200

    H["adb"] = adb_name
    M["adb"] = adb_name
    L["adb"] = adb_name

    # Keep only required columns
    H = H[["RP", "adb", "geometry"]]
    M = M[["RP", "adb", "geometry"]]
    L = L[["RP", "adb", "geometry"]]

    logging.info(f"{adb_name} clipped + standardized")

    # -------------------------
    # 2. HIERARCHICAL OVERLAY
    # -------------------------

    # M - H
    M_clean = gpd.overlay(M, H, how="difference", keep_geom_type=False)

    HM = gpd.GeoDataFrame(
        pd.concat([H, M_clean], ignore_index=True),
        crs=H.crs
    )

    # L - (H + M)
    L_clean = gpd.overlay(L, HM, how="difference", keep_geom_type=False)

    # -------------------------
    # 3. FINAL MERGE
    # -------------------------
    final = gpd.GeoDataFrame(
        pd.concat([HM, L_clean], ignore_index=True),
        crs=H.crs
    )

    # Enforce schema again (safety after overlay)
    final = final[["RP", "adb", "geometry"]]

    logging.info(f"{adb_name} processed")

    return final

In [ ]:
delim_adb_sa = gpd.read_file(delim_adb_sa_file)
delim_adb_si = gpd.read_file(delim_adb_si_file)
delim_adb_ac = gpd.read_file(delim_adb_ac_file)

In [ ]:
for k in gdfs:
    gdfs[k] = gdfs[k].to_crs(delim_adb_ac.crs)

In [ ]:
target_crs = gdfs["H"].crs
delim_adb_ac = delim_adb_ac.to_crs(target_crs)
delim_adb_si = delim_adb_si.to_crs(target_crs)
delim_adb_sa = delim_adb_sa.to_crs(target_crs)

In [ ]:
gdf_ac = process_adb(delim_adb_ac, gdfs, "AC")
gdf_si = process_adb(delim_adb_si, gdfs, "SI")
gdf_sa = process_adb(delim_adb_sa, gdfs, "SA")

In [ ]:
final_all = gpd.GeoDataFrame(
    pd.concat([gdf_ac, gdf_si, gdf_sa], ignore_index=True),
    crs=gdf_ac.crs
)

In [ ]:
gdf_ac.to_file(r"/home/admin_climatecharted_com/data/ADB/ispra_adbac")
gdf_si.to_file(r"/home/admin_climatecharted_com/data/ADB/ispra_adbsi")
gdf_sa.to_file(r"/home/admin_climatecharted_com/data/ADB/ispra_adbsa")

final_all.to_file(r"/home/admin_climatecharted_com/data/ADB/ispra_adb_ac_si_sa")

In [ ]:
adb_merged = gpd.read_file(r"/home/admin_climatecharted_com/data/ADB/ADB_merged")

In [ ]:
final_ispra_adb = gpd.GeoDataFrame(
    pd.concat([adb_merged, final_all], ignore_index=True),
    crs=adb_merged.crs
)

In [ ]:
output_file = r"/home/admin_climatecharted_com/data/ADB/ispra_adb_2026"
final_ispra_adb.to_file(output_file)

### ADB PO overlay

In [ ]:
import rasterio

In [ ]:
with rasterio.open() as src:
    profile = src.profile.copy()
    data = src.read(1)
    
    # 1️⃣ Change existing nodata to -9999
    nodata_val = src.nodata if src.nodata is not None else -9999
    data[data == nodata_val] = -9999
    profile.update(nodata=-9999)
    
    # 2️⃣ Mask outside Italy
    mask_image, _ = mask(src, geoms, invert=False, filled=False)
    mask_image = mask_image[0].astype(bool)  # Convert 3D -> 2D, True inside Italy
    
    # Pixels outside Italy with value 0 → -9999
    outside_mask = ~mask_image
    data[outside_mask & (data == 0)] = -9999

# Save raster
with rasterio.open(output_file, 'w', **profile) as dst:
    dst.write(data, 1)